# Data Prep for Neural Network

In [14]:
import polars as pl

## Train Test Split

Here we perform the split into train, validation and test data, with a **split** of 70% train, 15% validation and 15% test data. The split is performed **random**. 

In [15]:
DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_COMMUNITY_AREA.parquet"
OUTPUT = "../data/train_test_data/"
TARGET_COL = "trip_count"

SEED = 42

In [16]:
df_split = (
    pl.scan_parquet(DATASET)
    .with_row_index("_row_id")
    .with_columns(
        (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
    )
)

train = (
    df_split
    .filter(pl.col("_split_bucket") < 70)
    .drop(["_row_id", "_split_bucket"])
)

val = (
    df_split
    .filter(
        (pl.col("_split_bucket") >= 70) &
        (pl.col("_split_bucket") < 85)
    )
    .drop(["_row_id", "_split_bucket"])
)

test = (
    df_split
    .filter(pl.col("_split_bucket") >= 85)
    .drop(["_row_id", "_split_bucket"])
)

total_count = df_split.select(pl.len()).collect().item()
train_count = train.select(pl.len()).collect().item()
val_count = val.select(pl.len()).collect().item()
test_count = test.select(pl.len()).collect().item()

print("Total:", total_count)
print("Train:", train_count, " Share: ", round(train_count / total_count,2))
print("Val:", val_count, " Share: ", round(val_count / total_count,2))
print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

train.sink_parquet(OUTPUT + "train.parquet")
val.sink_parquet(OUTPUT + "val.parquet")
test.sink_parquet(OUTPUT + "test.parquet")

Total: 1574265
Train: 1101002  Share:  0.7
Val: 236011  Share:  0.15
Test: 237252  Share:  0.15


In [6]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
0,2025-06-19 02:00:00,6,4,2,0.5,-0.866025,0.433884,-0.900969,0.5,0.866025,19.44,87.0,7.666667,8.666667,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,55
1,2025-06-22 17:00:00,6,7,17,0.5,-0.866025,-0.781831,0.62349,-0.965926,-0.258819,34.44,47.55,20.0,10.0,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,1,2100,2100.0,2100,2100,8.4,8.4,8.4,8.4,25.75,25.75,25.75,25.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.75,25.75,25.75,25.75,"""Cash""",30
2,2025-06-23 22:00:00,6,1,22,0.5,-0.866025,0.0,1.0,-0.5,0.866025,31.11,57.33,5.0,10.0,0.0,0,1,0,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,67
3,2025-06-24 09:00:00,6,2,9,0.5,-0.866025,0.781831,0.62349,0.707107,-0.707107,32.22,57.6,9.0,10.0,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,81
4,2025-06-25 13:00:00,6,3,13,0.5,-0.866025,0.974928,-0.222521,-0.258819,-0.965926,31.11,57.33,5.0,10.0,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,3,7308,2436.0,2268,2580,30.31,10.103333,8.41,12.1,87.0,29.0,26.0,31.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.333333,0.0,1.0,88.0,29.333333,26.0,31.75,"""Unknown""",82
5,2025-06-25 22:00:00,6,3,22,0.5,-0.866025,0.974928,-0.222521,-0.5,0.866025,26.11,74.02,3.0,10.0,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,13
6,2025-06-26 00:00:00,6,4,0,0.5,-0.866025,0.433884,-0.900969,0.0,1.0,26.67,71.64,6.0,10.0,0.0,0,1,0,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,50
7,2025-06-26 02:00:00,6,4,2,0.5,-0.866025,0.433884,-0.900969,0.5,0.866025,26.11,74.02,5.0,10.0,0.0,0,1,0,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1
8,2025-06-29 06:00:00,6,7,6,0.5,-0.866025,-0.781831,0.62349,1.0,6.1232e-17,25.56,71.43,3.0,10.0,0.0,0,0,1,0,0,0,11,38.0,10.0,13.0,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,29


## Feature Selection